In [1]:
%%capture
# !pip install -U peft transformers diffusers openai
# !pip install git+https://github.com/openai/CLIP.git
# # !git clone https://github.com/christophschuhmann/improved-aesthetic-predictor.git
# # import sys
# # sys.path.append("improved-aesthetic-predictor")

# !pip install diffusers --upgrade
# !pip install invisible_watermark transformers accelerate safetensors


# !pip uninstall -y scipy numpy
# !pip install -U --no-cache-dir numpy==1.26.4 scipy==1.11.4

%%capture
!pip uninstall -y numpy scipy diffusers transformers peft accelerate safetensors
!pip install -U --no-cache-dir "numpy==1.26.4" "scipy==1.11.4"
!pip install -U --no-cache-dir "torch" "torchvision" "torchaudio"
!pip install -U --no-cache-dir "diffusers" "transformers" "peft" "accelerate" "safetensors" "openai" "invisible_watermark"



In [2]:
from diffusers import DiffusionPipeline
# from aesthetic_predictor import AestheticPredictor
# from transformers import Blip2Processor, Blip2Model
from openai import OpenAI
import torch
# import clip
#from PIL import Image
import os

import base64
from io import BytesIO
import json
import time

# import torch.nn.functional as F

2026-01-06 09:20:18.991972: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767691219.295118      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767691219.384883      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767691220.199243      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767691220.199278      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767691220.199281      55 computation_placer.cc:177] computation placer alr

In [ ]:
my_token = "" # enter your api key here

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# clip_model, preprocess = clip.load("ViT-B/32", device=device)

In [5]:
# import time, random
# from openai import RateLimitError

# def call_with_backoff(fn, *, max_retries=8, base_sleep=2.0):
#     for attempt in range(max_retries):
#         try:
#             return fn()
#         except RateLimitError:
#             sleep = base_sleep * (2 ** attempt) + random.uniform(0, 0.5)
#             print(f"Rate limit. Sleep {sleep:.1f}s then retry ({attempt+1}/{max_retries})...")
#             time.sleep(sleep)
#     raise RuntimeError("Rate limit persists after retries.")


In [6]:
import json
import re
from openai import OpenAI

endpoint = "https://models.github.ai/inference"
model = "gpt-4.1-mini"

client = OpenAI(
    api_key=my_token,
)

SYSTEM = (
    "You are an evaluation planner for text-to-image generation using DPG-style scoring. "
    "Your job is to convert a prompt into (1) an optional enhanced prompt and (2) a set of atomic, visually-checkable constraints "
    "with importance weights for automatic evaluation."
)

def _extract_json_loose(text: str):
    """Try to extract the first JSON object from a messy LLM output."""
    if not text:
        return None
    # find first {...} block
    m = re.search(r"\{[\s\S]*\}", text)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None

def build_constraints(user_prompt: str, *, max_constraints=28, enhance=True):
    """
    Return:
    {
      "enhanced_prompt": "...",   # if enhance=True else original/translated prompt
      "constraints":[
         {"id":"c01","group":"global|entity|attribute|relation|negative","text":"...","weight":0.0..1.0},
         ...
      ]
    }
    """
    TEMPLATE = f"""
You will read USER_PROMPT and produce STRICT JSON ONLY (no markdown, no extra text).

<USER_PROMPT>
{user_prompt}

Output JSON schema:
{{
  "enhanced_prompt": "<string>",
  "constraints": [
    {{
      "id": "c01",
      "group": "global|entity|attribute|relation|negative",
      "text": "<atomic visually-checkable yes/no constraint>",
      "weight": <float 0.0..1.0>
    }}
  ]
}}

Rules:

Constraints requirements:
- Create between {max(18, max_constraints-10)} and {max_constraints} constraints.
- Constraints must be ATOMIC and visually checkable from ONE image.
- Cover groups:
  - global: time/place/overall scene, setting, atmosphere
  - entity: key people/objects present
  - attribute: clothing/material/color/specific traits that are visible
  - relation: spatial/action relations (holding, standing before, behind, riding, etc.)
  - negative: modern artifacts, wrong uniforms/objects, anachronisms implied by the prompt (if any)
- NO duplicate constraints, NO vague constraints like "high quality".
- Weighting:
  - Core subject/action/time-period: 0.85–1.00
  - Major secondary elements: 0.60–0.85
  - Minor details: 0.25–0.60
  - Negative constraints: 0.50–0.90 depending on importance
- Make constraints phrased as YES/NO checks, e.g. "A Vietnamese general is holding a raised sword."
"""

    # Prefer forcing JSON output if the endpoint supports it:
    # Some OpenAI-compatible endpoints support response_format. If yours errors, remove response_format block.
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": SYSTEM},
                {"role": "user", "content": TEMPLATE},
            ],
            temperature=0.2,
            response_format={"type": "json_object"},
        )
        text = response.choices[0].message.content
        data = json.loads(text)
    except Exception:
        # Fallback without response_format
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": SYSTEM},
                {"role": "user", "content": TEMPLATE},
            ],
            temperature=0.2,
        )
        text = response.choices[0].message.content
        data = _extract_json_loose(text)

    if not data:
        raise ValueError("LLM did not return valid JSON. Raw output:\n" + str(text))

    # ---- sanitize output ----
    enhanced_prompt = str(data.get("enhanced_prompt", "")).strip()
    constraints = data.get("constraints", [])
    if not isinstance(constraints, list):
        constraints = []

    clean = []
    for i, c in enumerate(constraints, start=1):
        if not isinstance(c, dict):
            continue
        gid = str(c.get("id", f"c{i:02d}")).strip() or f"c{i:02d}"
        group = str(c.get("group", "")).strip().lower()
        if group not in {"global","entity","attribute","relation","negative"}:
            # try to coerce
            group = "entity"
        text_c = str(c.get("text", "")).strip()
        if not text_c:
            continue
        w = c.get("weight", 0.5)
        try:
            w = float(w)
        except Exception:
            w = 0.5
        w = max(0.0, min(1.0, w))
        clean.append({"id": gid, "group": group, "text": text_c, "weight": w})

    # de-duplicate by text (case-insensitive)
    seen = set()
    dedup = []
    for c in clean:
        key = c["text"].lower()
        if key in seen:
            continue
        seen.add(key)
        dedup.append(c)

    # re-id to stable sequential c01..cNN
    for idx, c in enumerate(dedup, start=1):
        c["id"] = f"c{idx:02d}"

    return {"enhanced_prompt": enhanced_prompt, "constraints": dedup}


In [7]:
import base64, json, numpy as np
from io import BytesIO

# --- Helper: PIL -> data URL ---
def pil_to_data_url(img, fmt="PNG"):
    buf = BytesIO()
    img.save(buf, format=fmt)
    b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    return f"data:image/{fmt.lower()};base64,{b64}"

SYSTEM_VISION_EVAL = """
You are evaluating whether an image satisfies a list of constraints derived from a prompt.
Return STRICT JSON only.

Schema:
{
  "results":[
    {"id":"c01","pass":true|false,"confidence":0.0..1.0,"rationale":"<=18 words"},
    ...
  ]
}

Rules:
- Be strict: if uncertain, set pass=false with low confidence.
- confidence reflects certainty about pass/fail.
- rationale must be short and grounded in visible evidence.
"""

def eval_one_image_with_llm(img, constraints, model_vision=None):
    """
    constraints: list of {id, text, weight, group}
    returns: detail(list), score(float)
    """
    if model_vision is None:
        model_vision = model  # reuse your "openai/gpt-4.1"

    data_url = pil_to_data_url(img)

    payload = {
        "constraints": [{"id":c["id"], "text":c["text"]} for c in constraints]
    }

    # --- IMPORTANT: chat.completions with multimodal content ---
    resp = client.chat.completions.create(
        model=model_vision,
        messages=[
            {"role":"system","content":SYSTEM_VISION_EVAL},
            {"role":"user","content":[
                {"type":"text","text":json.dumps(payload, ensure_ascii=False)},
                {"type":"image_url","image_url":{"url":data_url}},
            ]}
        ],
        temperature=0.0,
    )
    time.sleep(0.2)

    raw = resp.choices[0].message.content
    # in case model returns extra text, try extract JSON
    try:
        out = json.loads(raw)
    except Exception:
        out = _extract_json_loose(raw)
        if out is None:
            raise ValueError("Vision model did not return JSON.\nRAW:\n" + str(raw))

    res_map = {r["id"]: r for r in out.get("results", []) if isinstance(r, dict) and "id" in r}

    num, den = 0.0, 0.0
    detail = []
    for c in constraints:
        r = res_map.get(c["id"], {"pass": False, "confidence": 0.0, "rationale": "missing"})
        w = float(c["weight"])
        p = 1.0 if bool(r.get("pass", False)) else 0.0
        conf = float(r.get("confidence", 0.0))
        conf = max(0.0, min(1.0, conf))
        num += w * p * conf
        den += w
        detail.append({
            "id": c["id"],
            "group": c["group"],
            "weight": w,
            "pass": bool(r.get("pass", False)),
            "confidence": conf,
            "text": c["text"],
            "rationale": r.get("rationale", "")
        })

    score = (num / den) if den > 0 else 0.0
    return detail, score

def rank_images_llm(images, prompt, max_constraints=28, enhance=False, model_text=None, model_vision=None):
    """
    1) LLM gen constraints + weight
    2) (optional) enhance prompt (không bắt buộc)
    3) LLM-vision chấm từng ảnh -> score
    """
    if model_text is None:
        model_text = model

    # 1) gen constraints + weight
    plan = build_constraints(prompt, max_constraints=max_constraints, enhance=enhance)
    enhanced_prompt = plan["enhanced_prompt"]
    constraints = plan["constraints"]

    print ("Numbers of constraints: ", len(constraints))
    print("Constraints:", constraints)

    # 2) score images
    scores = []
    details_all = []
    for i, img in enumerate(images):
        detail, s = eval_one_image_with_llm(img, constraints, model_vision=model_vision)
        scores.append(s)
        details_all.append(detail)
        print(f"img {i:02d} | DPG={s:.4f}")

    best_idx = int(np.argmax(scores))
    print("BEST IDX =", best_idx, "| BEST DPG =", scores[best_idx])

    return {
        "enhanced_prompt": enhanced_prompt,
        "constraints": constraints,
        "scores": scores,
        "best_idx": best_idx,
        "details": details_all
    }


In [8]:
prompts = ["Dinh Bo Linh standing at the front of his army, holding a sword, soldiers behind him with ancient Vietnamese banners, hills and rivers in the background, armored warriors preparing for battle, no text, no letters, no symbols",
           "The Trưng Sisters riding war elephants side by side, wearing ancient Vietnamese armor, soldiers following them, battlefield filled with flags and drums, mountains in the distance, no text, no letters, no symbols",
           "Ly Thai To standing on a platform near the Red River, officials and guards surrounding him, a dragon appearing above the river, ancient palaces and city structures behind, no text, no letters, no symbols",
           "Tran Hung Dao commanding from a riverbank, wooden stakes emerging from the river, enemy ships trapped among the stakes, Vietnamese soldiers attacking from boats, clouds and smoke over the battlefield, no text, no letters, no symbols",
           "Emperor Quang Trung riding at the head of his army, soldiers advancing through smoke and fire, broken enemy formations, banners waving, hills and villages in the background, no text, no letters, no symbols",
           "Le Loi holding a sword raised toward the sky, soldiers gathered around him, forested mountains behind, tents and weapons on the ground, morning mist covering the battlefield, no text, no letters, no symbols",
           "Ho Chi Minh standing on a stage at Ba Dinh Square, a large crowd in front of him, flags held by the people, surrounding buildings visible, clear sky above, no text, no letters, no symbols",
           "Vietnamese soldiers moving through t renches and hills, carrying weapons and supplies, artillery positioned on the mountains, smoke rising from the battlefield, dense forest around, no text, no letters, no symbols",
           "People gathered around a large bronze drum, dancers and musicians surrounding it, traditional clothing, boats and animals depicted on the drum surface, river and trees nearby, no text, no letters, no symbols",
           "The Vietnamese emperor seated on a throne inside a palace, mandarins standing in rows on both sides, guards near the columns, traditional palace architecture visible, no text, no letters, no symbols"
          ]

In [11]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from PIL import Image

# ========== CONFIG ==========
ROOT = Path("/kaggle/input/image-evaluation-from-text-to-image/image evaluation")  # đổi nếu path khác
MODEL_FOLDERS = ["FFusionXL", "SD-v1.5", "SD-xl", "qwen"]  # đúng theo cây thư mục của bạn
# số prompt = len(prompts) (ưu tiên theo list prompt trong code của bạn)
NUM_PROMPTS = len(prompts)

# nếu bạn muốn giới hạn số ảnh mỗi prompt để tiết kiệm token/cost:
MAX_IMAGES_PER_PROMPT = None  # None = lấy hết; ví dụ 8 = lấy tối đa 8 ảnh/folder

# có enhance prompt hay không (nếu bạn đang dùng pipeline LLM để "chuẩn hoá" prompt)
ENHANCE_PROMPT = False
MAX_CONSTRAINTS = 28

# optional: cache kết quả để chạy lại không tốn gọi API lần nữa
CACHE_DIR = Path("./dpg_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def _sorted_pngs(folder: Path):
    """Lấy danh sách .png theo thứ tự số: 0.png, 1.png,... nếu có; fallback sort chữ."""
    if not folder.exists():
        return []
    files = list(folder.glob("*.png"))
    if not files:
        return []
    def key(p: Path):
        m = re.match(r"(\d+)\.png$", p.name)
        return int(m.group(1)) if m else 10**9
    files.sort(key=lambda p: (key(p), p.name))
    return files


def eval_prompt_folder(prompt_text: str, folder: Path, *, prompt_idx: int, model_name: str):
    """
    Trả về dict:
      {
        "prompt_idx": i,
        "n_images": k,
        "img_scores": [...],
        "prompt_avg": float | None
      }
    """
    # đọc ảnh
    img_paths = _sorted_pngs(folder)
    if MAX_IMAGES_PER_PROMPT is not None:
        img_paths = img_paths[:MAX_IMAGES_PER_PROMPT]

    if len(img_paths) == 0:
        return {
            "prompt_idx": prompt_idx,
            "n_images": 0,
            "img_scores": [],
            "prompt_avg": None
        }

    # ====== CACHE constraints theo prompt (đỡ gọi LLM text nhiều lần) ======
    cons_cache_path = CACHE_DIR / f"constraints__p{prompt_idx}.json"
    if cons_cache_path.exists():
        constraints_pack = pd.read_json(cons_cache_path, typ="series").to_dict()
    else:
        constraints_pack = build_constraints(
            prompt_text,
            max_constraints=MAX_CONSTRAINTS,
            enhance=ENHANCE_PROMPT
        )
        # lưu cache
        pd.Series(constraints_pack).to_json(cons_cache_path, force_ascii=False)

    constraints = constraints_pack["constraints"]

    # ====== Eval từng ảnh ======
    scores = []
    for p in img_paths:
        img = Image.open(p).convert("RGB")

        # ====== CACHE score theo ảnh ======
        img_cache_path = CACHE_DIR / f"score__{model_name}__p{prompt_idx}__{p.stem}.json"
        if img_cache_path.exists():
            s = float(pd.read_json(img_cache_path, typ="series")["score"])
        else:
            detail, s = eval_one_image_with_llm(img, constraints, model_vision=model)
            pd.Series({"score": float(s)}).to_json(img_cache_path, force_ascii=False)

        scores.append(float(s))

    return {
        "prompt_idx": prompt_idx,
        "n_images": len(scores),
        "img_scores": scores,
        "prompt_avg": float(np.mean(scores)) if scores else None
    }


def eval_one_model(model_name: str):
    """
    Duyệt folder 0..NUM_PROMPTS-1, tính:
      - prompt_avg cho từng prompt (nếu có ảnh)
      - model_avg = mean(prompt_avg của các prompt có ảnh)
    """
    model_root = ROOT / model_name
    per_prompt = []
    for i in range(NUM_PROMPTS):
        folder_i = model_root / str(i)
        res_i = eval_prompt_folder(prompts[i], folder_i, prompt_idx=i, model_name=model_name)
        per_prompt.append(res_i)

    # chỉ lấy prompt có prompt_avg != None
    valid = [x["prompt_avg"] for x in per_prompt if x["prompt_avg"] is not None]
    model_avg = float(np.mean(valid)) if len(valid) else None
    n_valid_prompts = sum(1 for x in per_prompt if x["prompt_avg"] is not None)

    return {
        "model": model_name,
        "model_avg_dpg": model_avg,
        "valid_prompts": n_valid_prompts,
        "total_prompts": NUM_PROMPTS,
        "per_prompt": per_prompt
    }


# ========== RUN ALL MODELS ==========
all_results = []
for m in MODEL_FOLDERS:
    all_results.append(eval_one_model(m))

# bảng tổng hợp
summary_rows = []
for r in all_results:
    summary_rows.append({
        "model": r["model"],
        "DPG_avg_all_prompts": r["model_avg_dpg"],
        "valid_prompts": r["valid_prompts"],
        "total_prompts": r["total_prompts"],
    })

df_summary = pd.DataFrame(summary_rows).sort_values(
    by="DPG_avg_all_prompts", ascending=False, na_position="last"
)
df_summary


,model,DPG_avg_all_prompts,valid_prompts,total_prompts
3,qwen,0.806707,9,10
0,FFusionXL,0.686256,10,10
2,SD-xl,0.656452,10,10
1,SD-v1.5,0.586875,10,10
